In [1]:
import numpy as np
from glob import glob
import os
import skimage as sk
from skan import draw
from skan.csr import skeleton_to_csgraph
from skan import Skeleton, summarize
import napari
import matplotlib.pyplot as plt
import pandas as pd

## Create Skeletons!

In [2]:
cropped_imgs_50_files = sorted(glob(r"Image_Data\Oct_2025_300mm_exp\50_mM\Masked_cropped_images\*.tif"))
cropped_imgs_300_1_files = sorted(glob(r"Image_Data\Oct_2025_300mm_exp\300_mM_1_hr\Masked_cropped_images\*.tif"))
cropped_imgs_300_6_files = sorted(glob(r"Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Masked_cropped_images\*.tif"))

cropped_imgs_50 = list(map(sk.io.imread,cropped_imgs_50_files))
cropped_imgs_300_1 = list(map(sk.io.imread,cropped_imgs_300_1_files))
cropped_imgs_300_6 = list(map(sk.io.imread,cropped_imgs_300_6_files))

In [3]:
equalized_hist_50 = [sk.exposure.equalize_adapthist(img) for img in cropped_imgs_50]
equalized_hist_300_1 = [sk.exposure.equalize_adapthist(img) for img in cropped_imgs_300_1]
equalized_hist_300_6 = [sk.exposure.equalize_adapthist(img) for img in cropped_imgs_300_6]

c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 6622 fits in uint16
  return _convert(image, np.uint16, force_copy)
c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 7367 fits in uint16
  return _convert(image, np.uint16, force_copy)
c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 5835 fits in uint16
  return _convert(image, np.uint16, force_copy)
c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 8923 fits in uint16
  return _convert(image, np.uint16, force_copy)
c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting

In [4]:
#create binary images using Yen's method
threshold_yen_50 =[img > sk.filters.threshold_yen(img) for img in equalized_hist_50]
threshold_yen_300_1 =[img > sk.filters.threshold_yen(img) for img in equalized_hist_300_1]
threshold_yen_300_6 =[img > sk.filters.threshold_yen(img) for img in equalized_hist_300_6]

c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\filters\thresholding.py:466: RuntimeWarning: divide by zero encountered in log
  crit = np.log(((P1_sq[:-1] * P2_sq[1:]) ** -1) * (P1[:-1] * (1.0 - P1[:-1])) ** 2)


In [9]:
viewer = napari.view_image(cropped_imgs_50[20])
viewer.add_labels(threshold_yen_50[20])

E:\Temp\ipykernel_29796\31700049.py:1: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(cropped_imgs_50[20])


<Labels layer 'Labels' at 0x1f34c562e90>

In [16]:
#remove objects smaller than 30 pixels from binary images
filtered_objects_50 = [sk.morphology.remove_small_objects(img,min_size=50) for img in threshold_yen_50]
filtered_objects_300_1 = [sk.morphology.remove_small_objects(img,min_size=50) for img in threshold_yen_300_1]
filtered_objects_300_6 = [sk.morphology.remove_small_objects(img,min_size=50) for img in threshold_yen_300_6]

In [20]:
fill_small_holes_50 = [sk.morphology.remove_small_holes(img,area_threshold=50) for img in filtered_objects_50]
fill_small_holes_300_1 = [sk.morphology.remove_small_holes(img,area_threshold=50) for img in filtered_objects_300_1]
fill_small_holes_300_6 = [sk.morphology.remove_small_holes(img,area_threshold=50) for img in filtered_objects_300_6]

In [23]:
viewer = napari.view_image(cropped_imgs_300_1[7])
viewer.add_labels(threshold_yen_300_1[7], name="original")
viewer.add_labels(fill_small_holes_300_1[7], name='filled holes')

E:\Temp\ipykernel_29796\5499678.py:1: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(cropped_imgs_300_1[7])


<Labels layer 'filled holes' at 0x1f38969f160>

In [ ]:
#create labeled images from filtered binary images
labeled_objects_50 = [sk.morphology.label(img) for img in filtered_objects]
labeled_objects_300_1 = [sk.morphology.label(img) for img in filtered_objects]
labeled_objects_300_6 = [sk.morphology.label(img) for img in filtered_objects]

In [ ]:
#skeletonize labeled images
skeletons = [sk.morphology.skeletonize(img) for img in labeled_objects]

In [ ]:
#save skeleton images
files = sorted(glob("cropped_imgs/*.tif"))
path = "skeletons"
for i in range(len(skeletons)):
    name = os.path.basename(files[i])
    sk.io.imsave(os.path.join(path,name[:-4]+'_skeleton.tif'),skeletons[i])


In [ ]:
#save overlay images of skeletons on equalized hist images
save_path = 'skeletons'
for i in range(len(skeletons)):
    name = os.path.basename(files[i])
    fig,ax=plt.subplots()
    draw.overlay_skeleton_2d(equalized_hist[i],skeletons[i],image_cmap='Greys_r',dilate=1,axes=ax)
    fig.set_tight_layout(tight=True)
    plt.show(block=True)
    fig.savefig(os.path.join(save_path,name[:-4]+'_skeleton_overlay.png'),dpi=300)


### Analyze Skeletons of mito networks

In [ ]:
spacing_um = 0.07
branch_data = [summarize(Skeleton(skeleton,spacing=spacing_um),separator='_') for skeleton in skeletons]

In [ ]:
#assess dataframe
branch_data[9].tail()

In [ ]:
#quick visual of all the branch distances organized by branch type
branch_data.hist(column='branch_distance', by='branch_type', bins=100)

In [ ]:
#Quick visual of how the skeletons are labeled by branch type
draw.overlay_euclidean_skeleton_2d(equalized_hist[0], branch_data[0],
                                   skeleton_color_source='branch_type')

In [ ]:
files = sorted(glob("cropped_imgs/*.tif"))
names = [os.path.basename(file) for file in files]

In [ ]:
#add filename column to each dataframe and merge all dataframes into one
for name,df in zip(names,branch_data):
    df['filename'] = name

merged_df = pd.concat(branch_data)

In [ ]:
merged_df.to_csv('Output/skeleton_df.csv')

In [ ]:
import seaborn as sns

In [ ]:
#plot branch distances for junction to junction branches (branch_type 1) per condition; code from SKAN docs
#added a 'Condition' column based on filename
j2j=(merged_df[merged_df['branch_type']== 1].
     rename(columns={'branch_distance':
                     'branch distance (um)'}))
per_image = j2j.groupby('filename').median()
per_image['Condition'] = ['500 mM' if '500mm' in fn else '50 mM' for fn in per_image.index]

sns.stripplot(data=per_image,
              x='Condition',y='branch distance (um)',
              order=['50 mM','500 mM'],
              jitter=True)

In [ ]:
per_image.to_csv("Output/skeleton_df.csv")